In [ ]:
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report


SEED = 42

random.seed(SEED)
np.random.seed(SEED)

data = train.copy().reset_index(drop=True)
data.columns = data.columns.str.strip()

feature_cols = [
    col for col in data.columns if col not in ["shot_id", "id", "Shot", "label"]
]

X = data[feature_cols]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

print(f"학습: {len(X_train)}개 / 검증: {len(X_test)}개")

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_tr_values = imputer.fit_transform(X_train)
X_valid_values = imputer.transform(X_test)

X_tr = scaler.fit_transform(X_tr_values)
X_valid = scaler.transform(X_valid_values)

knn = KNeighborsClassifier(
    n_neighbors=7,
    weights="distance",
    metric="minkowski",
    p=2,
    n_jobs=-1,
)

knn.fit(X_tr, y_train)

pred_knn = knn.predict(X_valid)

print("\n[KNN 검증 결과]")
print(f"정확도: {accuracy_score(y_test, pred_knn):.4f}")
print(
    f"Macro F1: "
    f"{f1_score(y_test, pred_knn, labels=knn.classes_, average='macro', zero_division=0):.4f}"
)
print(
    classification_report(
        y_test,
        pred_knn,
        labels=knn.classes_,
        digits=4,
        zero_division=0,
    )
)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_jobs=1)),
    ]
)

param_grid = {
    "knn__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21, 31],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["minkowski"],
    "knn__p": [1, 2],
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

search.fit(X_train, y_train)

best_pipeline = search.best_estimator_

imputer = best_pipeline.named_steps["imputer"]
scaler = best_pipeline.named_steps["scaler"]
knn = best_pipeline.named_steps["knn"]

pred_knn = best_pipeline.predict(X_test)

print("\n[최적 파라미터]")
print(search.best_params_)
print(f"교차검증 Macro F1: {search.best_score_:.4f}")

print("\n[KNN 검증 결과]")
print(f"정확도: {accuracy_score(y_test, pred_knn):.4f}")
print(f"Macro F1: {f1_score(y_test, pred_knn, average='macro', zero_division=0):.4f}")
print(
    classification_report(
        y_test,
        pred_knn,
        digits=4,
        zero_division=0,
    )
)
